# Pulling an example conversation from the data

In [6]:
import os
import pandas as pd
import numpy as np
import re

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
avar_analysis_data = './4-POWER-LAW-SCALING/reports/all.csv'

information_analysis_data = './data/ckpts/'

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [19]:
avarparams = pd.read_csv(avar_analysis_data)
avarparams = avarparams.loc[avarparams['corpus'].isin(['CANDOR'])]
avarparams = avarparams.loc[(avarparams['b'] > -.55) & (avarparams['b'] < -.49)]
avarparams.head()

,corpus,cid,speaker,self,a,b,k
11737,CANDOR,CANDOR-11-e937313f-66b6-4473-91f0-ce4752207458,5f11fc790127d60008657bfe,True,0.000637,-0.532281,9453
11740,CANDOR,CANDOR-31-7dfedc6e-c3c9-4d43-a21c-ebf8a230d116,5ee6c849ecd49045280d2ad8,False,0.001248,-0.534371,19110
11746,CANDOR,CANDOR-7-58dabc5c-7cef-4a50-8881-07c6c8e0f656,5e792902c0da0d0cb5ee3de5,False,0.001744,-0.541374,8385
11762,CANDOR,CANDOR-16-d22f728d-d56e-48a0-b2bd-cf72172cdc8f,5ca6bbf13b5fcf00100996e9,False,0.000391,-0.498487,23859
11768,CANDOR,CANDOR-13-63e861bf-3c6c-434c-9099-c8d8ba694caf,5ef6d07be683903cd5ae171d,False,0.000999,-0.505837,11935


In [21]:
k = avarparams[['cid', 'speaker']].value_counts()
k = k[k >= 2]
k

cid                                             speaker                 
CANDOR-13-3f8e18be-c2a6-4f6f-b35f-eeb24fa4a587  5e8bfc63e53da90ec5673f11    2
CANDOR-29-cd047d60-0b00-4b46-8f15-892fa5471510  5dd406d9c23e0d3e6dacc034    2
CANDOR-5-d8c75271-827d-414b-b574-bf5dd113dea7   5c4de8f726093e000134ee6e    2
CANDOR-5-c66aadaf-0c35-4150-a66e-1b41bc359748   5a49d6a06d85f80001c25bc4    2
CANDOR-13-3b6c1e76-b603-4b55-adf8-a715c2d0e7ee  5e7bd30c451d18212b458afa    2
                                                                           ..
CANDOR-11-6cc56ecc-d29c-4bb3-bbd9-c67236c15b0c  5e3f6b251d15101c5251d8b7    2
CANDOR-7-ba51006e-0b13-4071-b19c-3f4ece8f670d   5e21d253c7fe654b45073bcf    2
CANDOR-11-4aab823d-a152-4133-9383-dfc65a4ee06c  5f12f20001f4ce0008767b19    2
CANDOR-24-80e6c6c6-de9b-42f5-8aba-b7471a744303  5cc34f64b4a9220015911c45    2
CANDOR-11-c67c4974-2d84-4c96-871a-cc8be0ef6ccc  596fea64a5c32a0001584755    2
Name: count, Length: 169, dtype: int64

In [22]:
CONVO_ID = 'CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec'

## Get information values per turn

In [26]:
fs = [f for f in grab_all_files(information_analysis_data) if bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f))]
len(fs)

1656

In [30]:
for f in fs:
    df=pd.read_parquet(f)
    if CONVO_ID in df['conversation_id'].values:
        print(f)
        break

./data/ckpts/rosen/lme-ready/CANDOR-19-29.parquet


In [28]:
df.head()

,conversation_id,corpus,x_speaker,y_speaker,x_turn_id,y_turn_id,nx,ny,aCoS,I,H3,self,tau,AVAR
0,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,1,427,4,0.169059,102.240889,104.201752,False,146,0.036068
1,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,3,427,18,0.119159,57.833445,86.521408,False,146,0.020027
2,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,5,427,19,0.120419,52.638812,92.899521,False,146,0.014167
3,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,7,427,4,0.146527,76.663608,101.299423,False,146,0.008563
4,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,9,427,14,0.141406,76.112685,95.756920,False,146,0.072089


In [35]:
df['x_turn_id_'] = [int(v.split('-')[-1]) for v in df['x_turn_id']]

In [44]:
# df.to_parquet('/Users/zacharyrosen/Desktop/conversation-ex.parquet', engine='fastparquet', compression='snappy')

## Pulling the actual conversation

In [37]:
conversation_doc = 'CANDOR-19-29.parquet'
conversations_path = '/Users/zacharyrosen/pGPU/d/shapeoflang2/to_rosen'

In [38]:
fs = grab_all_files(conversations_path, conversation_doc)
fs

['/Users/zacharyrosen/pGPU/d/shapeoflang2/to_rosen/CANDOR-19-29.parquet']

In [39]:
convo = pd.read_parquet(fs[0])

In [40]:
convo.head()

,x_turn_id,x_utterance,x_speaker,x_start,x_stop,x_delta,conversation_id,x_race,self,y_turn_id,y_utterance,y_speaker,y_start,y_stop,y_delta,y_race
0,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,5ee9201596dd421333942d06,7.64,270.66,263.02,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,white,False,1,All right.,5f91a4aeb9f8bc031e8c27c9,270.94,273.55,2.61,black_or_african_american
1,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,5ee9201596dd421333942d06,7.64,270.66,263.02,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,white,False,3,"With. Mhm. All right. Yeah, I'm flowing, wow.",5f91a4aeb9f8bc031e8c27c9,274.14,280.96,6.82,black_or_african_american
2,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,5ee9201596dd421333942d06,7.64,270.66,263.02,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,white,False,5,Oh hi. Uh I mean how do you mean I was doing g...,5f91a4aeb9f8bc031e8c27c9,281.34,300.36,19.02,black_or_african_american
3,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,5ee9201596dd421333942d06,7.64,270.66,263.02,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,white,False,7,Alright equipment.,5f91a4aeb9f8bc031e8c27c9,300.94,302.65,1.71,black_or_african_american
4,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,5ee9201596dd421333942d06,7.64,270.66,263.02,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,white,False,9,Alright. Cool print. Yeah. Uh huh.,5f91a4aeb9f8bc031e8c27c9,305.94,310.76,4.82,black_or_african_american


In [42]:
dfc = pd.merge(
    left=df, left_on=['x_turn_id_', 'y_turn_id'],
    right=convo[['x_utterance', 'y_utterance', 'x_turn_id', 'y_turn_id']], right_on=['x_turn_id', 'y_turn_id'],
    how='left'
)
dfc.isna().sum()

conversation_id       0
corpus                0
x_speaker             0
y_speaker             0
x_turn_id_x           0
y_turn_id             0
nx                    0
ny                    0
aCoS                  0
I                     0
H3                    0
self                  0
tau                   0
AVAR               1156
x_turn_id_            0
x_utterance           0
y_utterance           0
x_turn_id_y           0
dtype: int64

In [43]:
dfc.head()

,conversation_id,corpus,x_speaker,y_speaker,x_turn_id_x,y_turn_id,nx,ny,aCoS,I,H3,self,tau,AVAR,x_turn_id_,x_utterance,y_utterance,x_turn_id_y
0,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,1,427,4,0.169059,102.240889,104.201752,False,146,0.036068,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,All right.,0
1,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,3,427,18,0.119159,57.833445,86.521408,False,146,0.020027,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,"With. Mhm. All right. Yeah, I'm flowing, wow.",0
2,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,5,427,19,0.120419,52.638812,92.899521,False,146,0.014167,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,Oh hi. Uh I mean how do you mean I was doing g...,0
3,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,7,427,4,0.146527,76.663608,101.299423,False,146,0.008563,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,Alright equipment.,0
4,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec,CANDOR-19-29,5ee9201596dd421333942d06,5f91a4aeb9f8bc031e8c27c9,CANDOR-29-5a69b66a-61bc-45b5-98bd-ffb14fb1e1ec-0,9,427,14,0.141406,76.112685,95.756920,False,146,0.072089,0,sure. Mhm. Mhm. What? Mhm. Mhm. Okay. Mhm. Yea...,Alright. Cool print. Yeah. Uh huh.,0


In [45]:
dfc.to_parquet(
    '/Users/zacharyrosen/Desktop/conversation-ex.parquet',
    engine='fastparquet', compression='snappy'
)